In [109]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict,cross_validate,train_test_split,StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import confusion_matrix,precision_score,recall_score,roc_auc_score

In [110]:
ROOT = Path.cwd().parent

RAW = ROOT / "data" / "processed" / "clean.csv"

df = pd.read_csv(RAW)

In [111]:
df["has_balance"] = (df["Balance"] > 0).astype(int)

df["age_group"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 45, 55, 65, 75, 95],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76+"],
).astype(str)



In [112]:
X = df.drop(columns='Exited')

y = df['Exited']

In [113]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4,stratify=y)

In [114]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [115]:
X_train_basic = X_train.drop(columns=["has_balance","age_group"])

In [119]:
dummy = DummyClassifier()

validation = cross_validate(
  dummy,
  X_train_basic,
  y_train,
  cv=cv,
  scoring='accuracy'
)
predictions = cross_val_predict(dummy,X_train_basic,y_train,cv=cv)

print("basline test score:",validation['test_score'].mean())

print("precision",precision_score(y_train,predictions,zero_division=0))

print("recall",recall_score(y_train,predictions))


basline test score: 0.79625
precision 0.0
recall 0.0


In [120]:
preprocess = ColumnTransformer(
  [
    ("geo",OneHotEncoder(),["Geography"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","NumOfProducts","EstimatedSalary"]),
  ],
  remainder="passthrough"
)



In [121]:
pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_basic,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("basic logistic model test score:",validation['test_score'].mean())

predictions = cross_val_predict(pipe,X_train_basic,y_train,cv=cv)

print( )
print("confusion_matrix")
print(confusion_matrix(y_train,predictions))
print()

print("precision",precision_score(y_train,predictions))

print("recall",recall_score(y_train,predictions))

basic logistic model test score: 0.8115

confusion_matrix
[[6151  219]
 [1289  341]]

precision 0.6089285714285714
recall 0.20920245398773007


In [128]:
preprocess = ColumnTransformer(
  [
    ("geo",OneHotEncoder(),["Geography","NumOfProducts"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","EstimatedSalary"]),
  ],
  remainder="passthrough"
)

pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_basic,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("logistic with number of products as categry model test score:",validation['test_score'].mean())

predictions = cross_val_predict(pipe,X_train_basic,y_train,cv=cv)

print( )
print("confusion_matrix")
print(confusion_matrix(y_train,predictions))
print()

print("precision",precision_score(y_train,predictions))

print("recall",recall_score(y_train,predictions))

logistic with number of products as categry model test score: 0.8412499999999999

confusion_matrix
[[6115  255]
 [1015  615]]

precision 0.7068965517241379
recall 0.3773006134969325


In [129]:
X_train_1 = X_train.drop(columns="age_group") 

In [ ]:
preprocess = ColumnTransformer(
  [
    ("geo",OneHotEncoder(),["Geography","NumOfProducts"]),
    ("num",StandardScaler(),["CreditScore","Age","Tenure","Balance","EstimatedSalary"]),
  ],
  remainder="passthrough"
)


pipe = Pipeline(
  [("preprocess",preprocess),
  ("model",LogisticRegression(max_iter=1000))]
)

validation = cross_validate(
  pipe,
  X_train_1,
  y_train,
  cv=cv,
  scoring="accuracy",
  return_train_score=True
)

print("logistic model with has balnce test score:",validation['test_score'].mean())

predictions = cross_val_predict(pipe,X_train_1,y_train,cv=cv)

print( )
print("confusion_matrix")
print(confusion_matrix(y_train,predictions))
print()

print("precision",precision_score(y_train,predictions))

print("recall",recall_score(y_train,predictions))

logistic model with has balnce test score: 0.81225

confusion_matrix
[[6157  213]
 [1289  341]]

precision 0.6155234657039711
recall 0.20920245398773007
